# Smart City Sensor Data Analysis 2026
This notebook performs exploratory data analysis (EDA) on traffic, air quality, weather, and energy grid sensors.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style='whitegrid')

data_path = 'data/processed/'  # Change to 'data/raw/' or 'data/sample/' if needed

traffic = pd.read_csv(data_path + 'traffic_sensors_processed.csv')
air_quality = pd.read_csv(data_path + 'air_quality_sensors_processed.csv')
weather = pd.read_csv(data_path + 'weather_station_processed.csv')
energy = pd.read_csv(data_path + 'energy_grid_processed.csv')

print('Traffic Sensors Preview:')
display(traffic.head())

print('Air Quality Sensors Preview:')
display(air_quality.head())

print('Weather Stations Preview:')
display(weather.head())

print('Energy Grid Preview:')
display(energy.head())

## Descriptive Statistics

In [ ]:
print('Traffic Sensors Stats:')
display(traffic.describe())

print('Air Quality Sensors Stats:')
display(air_quality.describe())

print('Weather Stations Stats:')
display(weather.describe())

print('Energy Grid Stats:')
display(energy.describe())

## Merge Datasets by Timestamp and Location

In [ ]:
# Merge traffic and air quality
merged = pd.merge(traffic, air_quality, on=['timestamp','location'], how='outer')
# Merge weather
merged = pd.merge(merged, weather, on=['timestamp','location'], how='outer')
# Merge energy
merged = pd.merge(merged, energy, on=['timestamp','location'], how='outer')

print('Merged Dataset Preview:')
display(merged.head())

## Visualizations

In [ ]:
# Traffic count over time per zone
plt.figure(figsize=(10,5))
sns.lineplot(data=traffic, x='timestamp', y='traffic_count', hue='location')
plt.xticks(rotation=45)
plt.title('Traffic Count per Zone')
plt.xlabel('Timestamp')
plt.ylabel('Traffic Count')
plt.tight_layout()
plt.show()

# PM2.5 over time per zone
plt.figure(figsize=(10,5))
sns.lineplot(data=air_quality, x='timestamp', y='pm25', hue='location')
plt.xticks(rotation=45)
plt.title('Air Quality PM2.5 per Zone')
plt.xlabel('Timestamp')
plt.ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.show()

## Correlation Heatmap

In [ ]:
# Select numeric columns
numeric_cols = merged.select_dtypes(include='number')

# Compute correlation
corr = numeric_cols.corr()

# Plot heatmap
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Simple Anomaly Detection (Z-Score)

In [ ]:
# Compute z-scores
z_scores = numeric_cols.apply(stats.zscore)

# Identify anomalies: abs(z) > 3
anomalies = (z_scores.abs() > 3).any(axis=1)

print(f'Number of anomalies detected: {anomalies.sum()}')
display(merged[anomalies])